In [26]:
import os
import sys
import cv2
import torch
import random
import logging
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

sys.path.append(os.path.abspath("./vendor/SimSwap"))

from src.utils.config import Settings
from src.backends.insightface_backend import InsightFaceBackend
from models.fs_networks_512 import Generator_Adain_Upsample
from parsing_model.model import BiSeNet

logging.getLogger().setLevel(logging.ERROR)

# ---------------------------------------------------------
# ARCHITECTURE DU GÉNÉRATEUR ALLÉGÉ (L'Élève)
# ---------------------------------------------------------
class AdaIN(nn.Module):
    def __init__(self, style_dim, num_features):
        super().__init__()
        self.norm = nn.InstanceNorm2d(num_features, affine=False)
        self.fc = nn.Linear(style_dim, num_features * 2)

    def forward(self, x, style):
        h = self.fc(style)
        h = h.view(h.size(0), h.size(1), 1, 1)
        gamma, beta = torch.chunk(h, chunks=2, dim=1)
        return (1 + gamma) * self.norm(x) + beta

class AdaINResBlock(nn.Module):
    def __init__(self, channels, style_dim=512):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.adain1 = AdaIN(style_dim, channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.adain2 = AdaIN(style_dim, channels)

    def forward(self, x, style):
        residual = x
        out = self.conv1(x)
        out = self.adain1(out, style)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.adain2(out, style)
        return out + residual

class CustomLiteGenerator(nn.Module):
    def __init__(self, style_dim=512, num_blocks=4): 
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, padding=3),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.ReLU(inplace=True)
        )
        self.blocks = nn.ModuleList([AdaINResBlock(256, style_dim) for _ in range(num_blocks)])
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 3, kernel_size=7, padding=3),
            nn.Tanh() 
        )

    def forward(self, image, identity_vector):
        x = self.encoder(image)
        for block in self.blocks:
            x = block(x, identity_vector)
        return self.decoder(x)

# ---------------------------------------------------------
# ARCHITECTURE DU DISCRIMINATEUR (L'Expert)
# ---------------------------------------------------------
class PatchDiscriminator(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1)
        )

    def forward(self, x):
        return self.model(x)

In [27]:
# --- INITIALISATION D'INSIGHTFACE ---
print("Chargement d'InsightFace...")
settings = Settings(device="cuda" if torch.cuda.is_available() else "cpu")
recognizer = InsightFaceBackend(settings)
print("InsightFace prêt !")

class FaceSwapDataset(Dataset):
    def __init__(self, image_dir, recognizer):
        self.image_paths = list(Path(image_dir).rglob("*.png")) + list(Path(image_dir).rglob("*.jpg"))
        self.recognizer = recognizer
        print(f"Dataset prêt : {len(self.image_paths)} images trouvées.")
        
        self.transform = transforms.Compose([
            transforms.ToTensor(), 
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) 
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        
        img_pil = Image.open(img_path).convert('RGB')
        img_tensor = self.transform(img_pil)
        
        img_bgr = cv2.imread(str(img_path))
        faces = self.recognizer.detect(img_bgr)
        
        if not faces:
            # Sécurité si un visage n'est pas détecté
            return self.__getitem__(random.randint(0, len(self) - 1))
            
        main_face = sorted(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]), reverse=True)[0]
        identity_vector = torch.tensor(main_face.embedding, dtype=torch.float32)
        
        return img_tensor, identity_vector

# --- CHARGEMENT ---
dataset = FaceSwapDataset("processed_dataset_512", recognizer)
# Ajuster batch_size en fonction de la VRAM 
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, drop_last=True)

Chargement d'InsightFace...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
InsightFace prêt !
Dataset prêt : 29999 images trouvées.


In [31]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"L'entraînement s'effectuera sur : {device.upper()}")

# 1. L'ÉLÈVE et L'EXPERT
student_model = CustomLiteGenerator(num_blocks=4).to(device)
discriminator = PatchDiscriminator().to(device)

# 2. LE MAÎTRE
teacher_model = Generator_Adain_Upsample(input_nc=3, output_nc=3, latent_size=512, n_blocks=9)

path_simswap = "models/simswap/512/550000_net_G.pth" 
teacher_model.load_state_dict(torch.load(path_simswap, map_location=device),strict=False)
teacher_model = teacher_model.to(device)
teacher_model.eval()
for param in teacher_model.parameters():
    param.requires_grad = False
print("Maître SimSwap 512 chargé avec succès !")

# 3. LE MODÈLE DE SEGMENTATION (BiSeNet)
# 19 classes est le standard pour le dataset CelebAMask-HQ
bisenet_model = BiSeNet(n_classes=19) 

path_bisenet = "models/parsing/79999_iter.pth"
bisenet_model.load_state_dict(torch.load(path_bisenet, map_location=device))
bisenet_model = bisenet_model.to(device)
bisenet_model.eval()
for param in bisenet_model.parameters():
    param.requires_grad = False
print("Modèle BiSeNet chargé avec succès !")

# 4. JUGES ET OPTIMISEURS
criterion_GAN = nn.MSELoss()
criterion_pixel = nn.L1Loss()
criterion_distillation = nn.L1Loss() 

optimizer_G = optim.Adam(student_model.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

print("Tous les modèles et optimiseurs sont prêts pour le combat !")

L'entraînement s'effectuera sur : CPU
Maître SimSwap 512 chargé avec succès !


Downloading: "https://download.pytorch.org/models/resnet18-5c106cde.pth" to C:\Users\Thomas/.cache\torch\hub\checkpoints\resnet18-5c106cde.pth
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:21<00:00, 2.14MB/s]


Modèle BiSeNet chargé avec succès !
Tous les modèles et optimiseurs sont prêts pour le combat !


In [32]:
# --- PARAMÈTRES MLOPS ---
num_epochs = 50                 
patience_max = 5                
best_g_loss = float('inf')      
patience_counter = 0

checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)

print("🚀 Lancement de l'entraînement...")

for epoch in range(num_epochs):
    
    student_model.train()
    discriminator.train()
    
    for batch_idx, (target_images, identity_vectors) in enumerate(dataloader):
        
        target_images = target_images.to(device)
        identity_vectors = identity_vectors.to(device)
        
        # =========================================================
        # PHASE 1 : ENTRAÎNEMENT DU DISCRIMINATEUR
        # =========================================================
        optimizer_D.zero_grad()
        
        pred_real = discriminator(target_images)
        loss_D_real = criterion_GAN(pred_real, torch.ones_like(pred_real))
        
        fake_images = student_model(target_images, identity_vectors)
        pred_fake = discriminator(fake_images.detach())
        loss_D_fake = criterion_GAN(pred_fake, torch.zeros_like(pred_fake))
        
        loss_D = (loss_D_real + loss_D_fake) / 2
        loss_D.backward()
        optimizer_D.step()

        # =========================================================
        # PHASE 2 : ENTRAÎNEMENT DE L'ÉLÈVE
        # =========================================================
        optimizer_G.zero_grad()
        
        pred_fake_for_G = discriminator(fake_images)
        loss_G_GAN = criterion_GAN(pred_fake_for_G, torch.ones_like(pred_fake_for_G))
        loss_G_pixel = criterion_pixel(fake_images, target_images)
        
        # --- DISTILLATION MASQUÉE ---
        with torch.no_grad():
            # Le maître génère sa propre version parfaite du centre du visage
            teacher_fakes = teacher_model(target_images, identity_vectors)
            
            # --- LOGIQUE BISENET ---
            # BiSeNet s'attend souvent à des images normalisées différemment, mais 
            # avec la fonction de base, il sort une carte de probabilités
            bisenet_out = bisenet_model(target_images)[0]
            
            # On récupère la classe prédite pour chaque pixel (0=fond, 1=peau, 17=cheveux...)
            parsed = torch.argmax(bisenet_out, dim=1, keepdim=True)
            
            # On crée un masque binaire : 
            # Tout ce qui est entre 1 (peau) et 13 (bouche/menton) devient 1.
            # Tout le reste (cheveux, cou, fond, chapeau) devient 0.
            mask = ((parsed > 0) & (parsed < 14)).float() 
            
            
        masked_fake_images = fake_images * mask
        masked_teacher_fakes = teacher_fakes.detach() * mask
        
        loss_G_distill = criterion_distillation(masked_fake_images, masked_teacher_fakes)
        
        # --- APPLICATION DE LA PUNITION ---
        loss_G = loss_G_GAN + (10.0 * loss_G_pixel) + (15.0 * loss_G_distill)
        
        loss_G.backward()
        optimizer_G.step()
        
        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] Batch {batch_idx}/{len(dataloader)} | Loss D: {loss_D.item():.4f} | Loss G: {loss_G.item():.4f}")
            
    # =========================================================
    # EARLY STOPPING ET CHECKPOINTS
    # =========================================================
    current_g_loss = loss_G.item()
    if current_g_loss < best_g_loss:
        print(f" 🌟 Amélioration ! Loss G passe de {best_g_loss:.4f} à {current_g_loss:.4f}")
        best_g_loss = current_g_loss
        patience_counter = 0
        
        checkpoint_path = checkpoint_dir / f"lite_generator_best.pth"
        torch.save(student_model.state_dict(), checkpoint_path)
        print(f" 💾 Checkpoint : {checkpoint_path}")
    else:
        patience_counter += 1
        print(f" ⚠️ Patience : {patience_counter}/{patience_max}")
        if patience_counter >= patience_max:
            print("🛑 EARLY STOPPING : Le modèle n'apprend plus.")
            break

print("🎉 Entraînement terminé !")

🚀 Lancement de l'entraînement...
Epoch [1/50] Batch 0/7499 | Loss D: 0.4034 | Loss G: 7.4895
Epoch [1/50] Batch 10/7499 | Loss D: 0.2657 | Loss G: 4.4121


KeyboardInterrupt: 